In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
import datetime
import matplotlib.pyplot as plt

# Read HMS data

In [2]:
nhf_path = '/home/akagi/Documents/Data/txswift/nhf_1.1.3.gpkg'
nhf_flowlines = gpd.read_file(nhf_path, layer='flowpaths').set_index('fp_id')
nhf_subbasins = gpd.read_file(nhf_path, layer='divides').set_index('div_id')
div_to_fp_mapping = nhf_flowlines.reset_index().set_index('div_id')['fp_id']

In [3]:
nhd_path = '/home/akagi/Documents/Data/GIS/NHD_H_Texas_State_GDB/NHD_H_Texas_State_GDB.gdb'
nhd_flowlines = gpd.read_file(nhd_path, layer=18)
xmin, ymin, xmax, ymax = nhf_subbasins.to_crs(nhd_flowlines.crs).total_bounds
nhd_flowlines = nhd_flowlines.cx[xmin:xmax, ymin:ymax]
nhd_flowlines = nhd_flowlines.reset_index(drop=True)

/home/akagi/.local/lib/python3.11/site-packages/pyogrio/raw.py:196: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiLineString' is converted to 'MultiLineString Z'
  return ogr_read(


In [4]:
trinity_subbasins = gpd.read_file('/home/akagi/Documents/Data/txswift/hms/Subbasin.geojson')

Skipping field Last Modified Time: unsupported OGR type: 10


In [5]:
nhf_flowlines = nhf_flowlines.to_crs(trinity_subbasins.crs)
geom_array = nhf_flowlines.geometry

In [6]:
# Ensure multilinestrings are actually linestrings
#assert max(geom_array.apply(lambda x: len(x.geoms)).values) == 1

In [7]:
coords = geom_array.apply(lambda x: x.geoms[0]).apply(lambda y: y.coords.xy)
x = coords.str[0]
y = coords.str[1]
x0, x1 = x.str[0], x.str[-1]
y0, y1 = y.str[0], y.str[-1]
p0 = np.column_stack([x0.values, y0.values])
p1 = np.column_stack([x1.values, y1.values])
xy = np.column_stack([np.concatenate(x.values), np.concatenate(y.values)])

In [8]:
startpoints = [shapely.Point(xiyi) for xiyi in p0]
endpoints = [shapely.Point(xiyi) for xiyi in p1]

startpoints = gpd.GeoDataFrame(geometry=startpoints, crs=trinity_subbasins.crs)
endpoints = gpd.GeoDataFrame(geometry=endpoints, crs=trinity_subbasins.crs)

In [9]:
contains_startpoints = gpd.sjoin(trinity_subbasins, startpoints, predicate='contains').reset_index().set_index('index_right')
contains_endpoints = gpd.sjoin(trinity_subbasins, endpoints, predicate='contains').reset_index().set_index('index_right')

In [10]:
point_indices = np.unique(contains_startpoints.index.tolist() + contains_endpoints.index.tolist())

In [11]:
subbasin_connectivity = {}

for index in contains_startpoints.index:
    if (index in contains_startpoints.index) and (index in contains_endpoints.index):
        subbasin_index_start = int(contains_startpoints.loc[index, 'index'])
        subbasin_index_end = int(contains_endpoints.loc[index, 'index'])
        if subbasin_index_start != subbasin_index_end:
            subbasin_connectivity[subbasin_index_start] = subbasin_index_end

subbasin_connectivity = pd.Series(subbasin_connectivity)

In [12]:
startnodes = subbasin_connectivity.index
endnodes = subbasin_connectivity.values
is_endnode = np.isin(startnodes, endnodes)

In [13]:
headwater_subbasins = startnodes[~is_endnode]

In [14]:
trinity_subbasins['is_headwater'] = False
trinity_subbasins.loc[headwater_subbasins, 'is_headwater'] = True

In [15]:
trinity_subbasins.head(10)

,name,Last Modified Date,geometry,is_headwater
0,West_Fork_S010,4 February 2025,"MULTIPOLYGON (((-98.72482 33.39803, -98.72481 ...",True
1,West_Fork_S020,4 February 2025,"POLYGON ((-98.46995 33.51451, -98.46994 33.514...",False
2,West_Fork_S030,4 February 2025,"POLYGON ((-98.46366 33.42812, -98.46365 33.427...",False
3,West_Fork_S040,4 February 2025,"POLYGON ((-98.3915 33.46995, -98.39149 33.4696...",False
4,West_Fork_S050,4 February 2025,"POLYGON ((-98.38202 33.51099, -98.38201 33.510...",False
5,West_Fork_S060,4 February 2025,"POLYGON ((-98.30609 33.39511, -98.30606 33.394...",True
6,West_Fork_S070,4 February 2025,"MULTIPOLYGON (((-98.23513 33.46934, -98.2348 3...",True
7,West_Fork_S080,4 February 2025,"MULTIPOLYGON (((-98.25387 33.31821, -98.25355 ...",False
8,West_Fork_S090,4 February 2025,"POLYGON ((-98.23646 33.47039, -98.23645 33.470...",True
9,West_Fork_S100,4 February 2025,"POLYGON ((-98.15081 33.3339, -98.15113 33.3338...",False
